In [1]:
from pinecone import Pinecone

# Initialize the client
pc = Pinecone(api_key="pcsk_3Z18nd_New68WcRtj3Uybw1EaFKUvrPfg8Tp2AtbcD8Gdovfv6kPsnM7kkzJqt1Dqdaec8")

# Connect to your specific index
index_name = "google-5days-vibecoding"
index = pc.Index(index_name)

# Verify connection by getting index stats
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 532}},
 'total_vector_count': 532,
 'vector_type': 'dense'}


To use your `google-5days-vibecoding` index for Retrieval-Augmented Generation (RAG) in a Jupyter Notebook, you need to combine your vector database with an **embedding model** (to turn text into numbers) and an **LLM** (to generate the final answer).

The most common way to do this is using the [LangChain](https://python.langchain.com/) framework, which simplifies the integration.

### 1. Prerequisites

You will need your Pinecone API key and an API key for your chosen LLM (e.g., OpenAI, Anthropic, or Google Gemini). Install the necessary libraries:

```python
!pip install langchain-pinecone langchain-openai langchain-community

```

### 2. Basic RAG Workflow in Jupyter

Once connected to your index, the RAG flow generally follows these steps:

1. **Initialize Embeddings:** Ensure you use the **same** model that was used to create the vectors in your index originally.
2. **Connect to the VectorStore:** Use LangChain’s `PineconeVectorStore` to interface with your existing index.
3. **Create a Retriever:** Convert the vector store into a retriever object.
4. **Chain to LLM:** Use a RAG chain to pass the retrieved context to your LLM.

```python
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA

# 1. Initialize the embedding model (Match the one you used for indexing!)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. Connect to your existing index
index_name = "google-5days-vibecoding"
vectorstore = PineconeVectorStore(
    index_name=index_name, 
    embedding=embeddings,
    pinecone_api_key="YOUR_PINECONE_API_KEY"
)

# 3. Create a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 4. Set up the RAG chain
llm = ChatOpenAI(model="gpt-4o")
qa_chain = RetrievalQA.from_chain_type(llm, retriever=retriever)

# 5. Ask a question
question = "What is the 7-Pillar Security Architecture?"
result = qa_chain.invoke({"query": question})
print(result["result"])

```

### 3. Key Considerations for Your Data

Looking at your index browser, your data contains specific fields like `source`, `page`, and `text`.

* **Context Quality:** Because your documents (like *Vibe Coding Agent Security and Evaluation_Day_4.pdf*) are indexed, the `text` field will be automatically retrieved as context.
* **Metadata Filtering:** If you want to restrict searches to specific documents, you can add metadata filters:
```python
retriever = vectorstore.as_retriever(
    search_kwargs={
        "filter": {"source": "documents\\Vibe Coding Agent Security and Evaluation_Day_4.pdf"}
    }
)

```



### Alternative: Pinecone Assistant

If you don't want to manage embedding models or RAG chains manually, you can use the **[Pinecone Assistant API](https://www.pinecone.io/learn/pinecone-assistant/)**. This service handles the RAG logic for you. You simply point it at your data, and it provides a conversational interface for querying that specific information.

Would you like to try a specific implementation (e.g., using LangChain or perhaps just a raw Python script to test retrieval first)?

api_key = "pcsk_3Z18nd_New68WcRtj3Uybw1EaFKUvrPfg8Tp2AtbcD8Gdovfv6kPsnM7kkzJqt1Dqdaec8"
index_name = "google-5-days-vibecoding"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

pinecone_db = PineconeVectorStore(
    pinecone_instance=Pinecone(api_key=api_key),
    index_name=index_name,
    embedding_function=embeddings,
    # embeddding_dimension=384
)
retriever = pinecone_db.as_retriever(search_kwargs={"k": 3})

chat_history = []

qa = ConversationalRetrievalChain.from_llm(
    llm = llm,
    retriever = retriever,
    return_source_documents=True,
    condense_question_prompt=prompt,
)


In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_classic.chains import ConversationalRetrievalChain
from flask import Flask, request, jsonify,render_template
# from pinecnone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

prompt = PromptTemplate.from_template(  
    """
    ## A Professional Experienced AgenticAi/AI Teacher
    You are Teacher from Google 5 Days VibeCodes. You are a helpful assistant that helps students learn coding and programming concepts. You will answer the questions based on the context provided. 
    - Ask the user for clarification if the question is ambiguous or unclear.
    - Provide clear and concise explanations.
    - Use examples and analogies to help explain complex concepts.
    - Encourage the user to ask follow-up questions if they need further clarification.
    - Retrieve relevant information from the context to support your answers.
    - Your answers should be accurate, informative, and easy to understand related to retrieved context.

    ## Use this chat_history to answer the question. If the context is not enough, you can answer based on your knowledge. If you don't know the answer, just say "I don't know". Do not try to make up an answer.
    {chat_history}

    ## User Question: {question}
    ## Assistant:"""

)

from pinecone import Pinecone
import os
api_key = "pcsk_3Z18nd_New68WcRtj3Uybw1EaFKUvrPfg8Tp2AtbcD8Gdovfv6kPsnM7kkzJqt1Dqdaec8"

os.environ['PINECONE_API_KEY'] = api_key
# Initialize the Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key="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"
)


# To this:
index_name = "google-5days-vibecoding"


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

pinecone_db = PineconeVectorStore(
    pinecone_api_key=api_key,
    index_name=index_name,
    embedding=embeddings,
    # embeddding_dimension=384
)
retriever = pinecone_db.as_retriever(search_kwargs={"k": 3})

chat_history = []

qa = ConversationalRetrievalChain.from_llm(
    llm = llm,
    retriever = retriever,
    return_source_documents=True,
    condense_question_prompt=prompt,
)

qa.invoke({'question': 'what is agentic Ai?', 'chat_history': chat_history})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'question': 'what is agentic Ai?',
 'chat_history': [],
 'answer': 'Agentic AI refers to a type of artificial intelligence (AI) that is designed to act as an autonomous agent, capable of making decisions and taking actions on its own. In the context of "agentic engineering," it implies a more disciplined and structured approach to integrating AI into software development processes.\n\nAgentic AI typically involves the use of advanced machine learning algorithms, such as multi-agent systems, that enable the AI to interact with its environment, make decisions, and adapt to new situations. This type of AI is often characterized by its ability to:\n\n1. Learn from experience and improve its performance over time\n2. Reason and make decisions based on complex data and rules\n3. Interact with humans and other AI systems in a collaborative or autonomous manner\n4. Adapt to changing environments and circumstances\n\nIn the context of software development, agentic AI can be used to automate re